In [1]:
!pip install pynrrd

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import models, layers
import nrrd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

################################################################################
# 1) Load the Excel file and create the DataFrame (same as your original code)
################################################################################

excel_path = "/kaggle/input/my-lung-dataset/my_lung_dataset/dataset_lung.xlsx"
data_dir = "/kaggle/input/my-lung-dataset/my_lung_dataset/Train"

df = pd.read_excel(excel_path)

nodule_paths = []
labels_binary = []

for idx, row in df.iterrows():
    nodule_filename = row["Nodule"]       # e.g. "pat1_nodule.nrrd"
    tumor_class = row["TumorClass"]       # e.g. 1..5
    full_nodule_path = os.path.join(data_dir, nodule_filename)
    
    # [1,2,3] => 0 (benign), [4,5] => 1 (malignant)
    if tumor_class in [1,2,3]:
        label_bin = 0
    else:
        label_bin = 1
    
    nodule_paths.append(full_nodule_path)
    labels_binary.append(label_bin)

data_nodules = pd.DataFrame({
    "nodule_path": nodule_paths,
    "label": labels_binary
})

print("Total nodule images:", len(data_nodules))
data_nodules.head()

################################################################################
# 2) Define image loading & dataset creation (same logic as your original code)
################################################################################

IMG_SIZE = (128, 128)

def load_nrrd_image(nrrd_path, target_size=IMG_SIZE):
    volume_data, header = nrrd.read(nrrd_path)
    
    if volume_data.ndim == 3:
        if volume_data.shape[-1] == 1:
            volume_data = volume_data[..., 0]
        elif volume_data.shape[0] == 1:
            volume_data = volume_data[0, ...]
    
    volume_data = volume_data.astype(np.float32)
    volume_data = tf.image.resize(volume_data[..., tf.newaxis], target_size)
    
    # Convert to 3 channels (for pretrained networks):
    volume_data = tf.image.grayscale_to_rgb(volume_data)
    return volume_data.numpy()

def create_dataset(df, batch_size=8, shuffle=False):
    paths = df["nodule_path"].tolist()
    labels = df["label"].tolist()
    
    def gen():
        for p, l in zip(paths, labels):
            img = load_nrrd_image(p, target_size=IMG_SIZE)
            yield img, l

    dataset = tf.data.Dataset.from_generator(
        gen,
        output_types=(tf.float32, tf.int32),
        output_shapes=((IMG_SIZE[0], IMG_SIZE[1], 3), ())
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df))
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


################################################################################
# 3) Create a Transfer Learning Model (MobileNetV2 + Fine Tuning)
################################################################################

def create_transfer_model(input_shape=(128,128,3), fine_tune_at=None):
    """
    Builds a model with a MobileNetV2 backbone (pretrained on ImageNet).
    If `fine_tune_at` is None, we freeze the entire base model (only train the new head).
    If `fine_tune_at` is an integer, we'll unfreeze layers after that index for fine-tuning.
    """
    
    # 1) Load MobileNetV2 with pretrained weights
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape, 
        include_top=False, 
        weights='imagenet'
    )
    
    # 2) Choose how many layers to unfreeze (fine tuning)
    if fine_tune_at is None:
        # Freeze entire base model
        base_model.trainable = False
    else:
        # Unfreeze from 'fine_tune_at' layer onwards
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False
        for layer in base_model.layers[fine_tune_at:]:
            layer.trainable = True
    
    # 3) Build a "head" on top of base_model
    model = tf.keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    # 4) Compile the model
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    return model


################################################################################
# 4) Perform Cross Validation (StratifiedKFold) for a robust performance estimate
################################################################################

N_FOLDS = 5
BATCH_SIZE = 8
EPOCHS = 10  # You can adjust

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

X = data_nodules["nodule_path"].values
y = data_nodules["label"].values

fold_accuracies = []
fold_conf_matrices = []

# Callbacks
patience_es = 5
patience_lr = 3
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=patience_es, 
                                     restore_best_weights=True, 
                                     monitor='val_loss', 
                                     mode='min'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, 
                                         patience=patience_lr, 
                                         min_lr=1e-5,
                                         monitor='val_loss',
                                         mode='min')
]

fold_num = 1
for train_index, val_index in skf.split(X, y):
    print(f"\n=== Fold {fold_num}/{N_FOLDS} ===")
    
    df_train_fold = data_nodules.iloc[train_index].reset_index(drop=True)
    df_val_fold   = data_nodules.iloc[val_index].reset_index(drop=True)
    
    # Optional: compute class weights in this fold
    train_counts = df_train_fold["label"].value_counts()
    benign_count = train_counts.get(0, 0)
    malignant_count = train_counts.get(1, 0)
    total = benign_count + malignant_count
    weight_for_0 = (1 / benign_count) * (total / 2.0)
    weight_for_1 = (1 / malignant_count) * (total / 2.0)
    fold_class_weights = {0: weight_for_0, 1: weight_for_1}
    
    ds_train_fold = create_dataset(df_train_fold, batch_size=BATCH_SIZE, shuffle=True)
    ds_val_fold   = create_dataset(df_val_fold,   batch_size=BATCH_SIZE, shuffle=False)
    
    # Create the transfer model
    # Example: freeze entire base (no fine-tuning):
    # transfer_model = create_transfer_model(input_shape=(128,128,3), fine_tune_at=None)
    # or partially unfreeze from halfway in the MobileNetV2:
    transfer_model = create_transfer_model(input_shape=(128,128,3), fine_tune_at=100)
    
    # Train
    history = transfer_model.fit(
        ds_train_fold,
        epochs=EPOCHS,
        validation_data=ds_val_fold,
        callbacks=callbacks,
        class_weight=fold_class_weights,
        verbose=1
    )
    
    # Evaluate on fold's validation data
    val_loss, val_acc = transfer_model.evaluate(ds_val_fold, verbose=0)
    print(f"Fold {fold_num} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # Predictions & confusion matrix
    val_preds = transfer_model.predict(ds_val_fold).ravel()
    val_preds = (val_preds > 0.5).astype(int)
    
    val_true = []
    for _, lbl_batch in ds_val_fold:
        val_true.extend(lbl_batch.numpy())
    
    cm = confusion_matrix(val_true, val_preds)
    fold_conf_matrices.append(cm)
    
    print("Confusion Matrix:\n", cm)
    print(classification_report(val_true, val_preds,
          target_names=["Benign (0)", "Malignant (1)"]))
    
    fold_accuracies.append(val_acc)
    fold_num += 1

print("\n=== Cross Validation Results ===")
mean_acc = np.mean(fold_accuracies)
std_acc  = np.std(fold_accuracies)
print(f"Mean Validation Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

# Optionally: you can sum up confusion matrices or just inspect them individually.
# aggregated_cm = sum(fold_conf_matrices)  # sums each cell across folds
# print("Aggregated Confusion Matrix:\n", aggregated_cm)
#
# plt.figure(figsize=(5,4))
# sns.heatmap(aggregated_cm,
#             annot=True, fmt='d', cmap='Blues',
#             xticklabels=["Benign (0)", "Malignant (1)"],
#             yticklabels=["Benign (0)", "Malignant (1)"])
# plt.title("Confusion Matrix - Aggregated Over All Folds")
# plt.show()


Total nodule images: 2363

=== Fold 1/5 ===
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
    237/Unknown 43s 40ms/step - accuracy: 0.6560 - loss: 0.7441

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


237/237 ━━━━━━━━━━━━━━━━━━━━ 51s 76ms/step - accuracy: 0.6560 - loss: 0.7438 - val_accuracy: 0.7590 - val_loss: 2.3851 - learning_rate: 0.0010
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7552 - loss: 0.5837 - val_accuracy: 0.7696 - val_loss: 0.5574 - learning_rate: 0.0010
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7477 - loss: 0.5633 - val_accuracy: 0.6850 - val_loss: 0.6630 - learning_rate: 0.0010
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8070 - loss: 0.5317 - val_accuracy: 0.6934 - val_loss: 1.0150 - learning_rate: 0.0010
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8185 - loss: 0.4787 - val_accuracy: 0.5814 - val_loss: 0.8137 - learning_rate: 0.0010
Epoch 6/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8188 - loss: 0.4071 - val_accuracy: 0.6554 - val_loss: 0.7900 - learning_rate: 1.0000e-04
Epoch 7/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8497 - loss: 0.3718 -

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


237/237 ━━━━━━━━━━━━━━━━━━━━ 34s 53ms/step - accuracy: 0.5940 - loss: 0.8597 - val_accuracy: 0.7865 - val_loss: 1.1136 - learning_rate: 0.0010
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7214 - loss: 0.5872 - val_accuracy: 0.7949 - val_loss: 0.5766 - learning_rate: 0.0010
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7552 - loss: 0.5557 - val_accuracy: 0.5011 - val_loss: 1.2524 - learning_rate: 0.0010
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8060 - loss: 0.5186 - val_accuracy: 0.3002 - val_loss: 1.6826 - learning_rate: 0.0010
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8231 - loss: 0.4595 - val_accuracy: 0.7611 - val_loss: 0.5795 - learning_rate: 0.0010
Fold 2 - Val Loss: 1.1136, Val Acc: 0.7865
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 57ms/step
Confusion Matrix:
 [[357   2]
 [ 99  15]]
               precision    recall  f1-score   support

   Benign (0)       0.78      0.99      0.88       359
Malignant

...

237/237 ━━━━━━━━━━━━━━━━━━━━ 33s 57ms/step - accuracy: 0.6273 - loss: 0.8331 - val_accuracy: 0.7585 - val_loss: 3.2023 - learning_rate: 0.0010
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7498 - loss: 0.6257 - val_accuracy: 0.7860 - val_loss: 0.5470 - learning_rate: 0.0010
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.7807 - loss: 0.5488 - val_accuracy: 0.6144 - val_loss: 0.7574 - learning_rate: 0.0010
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.7970 - loss: 0.5201 - val_accuracy: 0.7733 - val_loss: 0.5382 - learning_rate: 0.0010
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.8193 - loss: 0.4878 - val_accuracy: 0.8008 - val_loss: 0.5092 - learning_rate: 0.0010
Fold 4 - Val Loss: 3.2023, Val Acc: 0.7585
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step
Confusion Matrix:
 [[358   0]
 [114   0]]
               precision    recall  f1-score   support

   Benign (0)       0.76      1.00      0.86       358
Malignant

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1/10
    237/Unknown 28s 32ms/step - accuracy: 0.6005 - loss: 0.8247

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


237/237 ━━━━━━━━━━━━━━━━━━━━ 32s 49ms/step - accuracy: 0.6008 - loss: 0.8241 - val_accuracy: 0.7881 - val_loss: 1.7517 - learning_rate: 0.0010
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7240 - loss: 0.5561 - val_accuracy: 0.8030 - val_loss: 0.5498 - learning_rate: 0.0010
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.7685 - loss: 0.5452 - val_accuracy: 0.4216 - val_loss: 0.9831 - learning_rate: 0.0010
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.7644 - loss: 0.5579 - val_accuracy: 0.3729 - val_loss: 2.0344 - learning_rate: 0.0010
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7705 - loss: 0.4959 - val_accuracy: 0.5000 - val_loss: 1.4855 - learning_rate: 0.0010
Fold 5 - Val Loss: 1.7517, Val Acc: 0.7881
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step
Confusion Matrix:
 [[356   2]
 [ 98  16]]
               precision    recall  f1-score   support

   Benign (0)       0.78      0.99      0.88       358
Malignant